# VM.AI Parser Training

**Enable GPU first:**
- Runtime → Change runtime type → GPU (T4) → Save

Uses **uv** for fast package installation.

In [ ]:
# Run src/parser/package_colab.py, it will package in colab.zip all files needed for this notebook
!unzip -o colab.zip

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!python train.py --mode both

Config loaded for mode: both
  Epochs: 8
  LR: 5e-06
  Batch: 8
  Grad Acc: 16
Device : cuda
Mode   : both
Epochs : 8
Resume : False  LR: 2e-05
Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 11 files: 100% 11/11 [00:34<00:00,  3.15s/it]
Download complete: : 4.46GB [00:34, 128MB/s] 
Starting from base model...
Loading weights: 100% 257/257 [00:00<00:00, 309.06it/s, Materializing param=shared.weight]
Real examples loaded: 100
No specific data file found — skipping
Train: 9180  |  Test: 1020
Map: 100% 9180/9180 [00:06<00:00, 1327.23 examples/s]
Map: 100% 1020/1020 [00:00<00:00, 1969.44 examples/s]
Starting training...
  → Starting fresh (no checkpoints found)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in curren

In [6]:
# Save to Drive
!mkdir -p /content/drive/MyDrive/vmai/models
!cp -r /content/models/finetuned_parser /content/drive/MyDrive/vmai/models/
print('Saved to Google Drive')

Saved to Google Drive


## Test Model (Chat Interface)

In [7]:
import os, shutil, re, zipfile
from google.colab import files as colab_files

model_dir = "/content/models/finetuned_parser"
if not os.path.exists(model_dir):
    print(f"Error: {model_dir} not found")
else:
    # Find latest checkpoint (optional, include for training continuity)
    checkpoints = []
    for d in os.listdir(model_dir):
        m = re.match(r"checkpoint-(\d+)", d)
        if m:
            checkpoints.append((int(m.group(1)), d))

    latest_ckpt = max(checkpoints, key=lambda x: x[0])[1] if checkpoints else None
    if latest_ckpt:
        print(f"Latest checkpoint: {latest_ckpt}")

    zip_name = "finetuned_parser.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in os.listdir(model_dir):
            if item.startswith(".cache"):
                continue
            # Skip older checkpoints to keep zip small
            if item.startswith("checkpoint-") and item != latest_ckpt:
                continue
            item_path = os.path.join(model_dir, item)
            if os.path.isfile(item_path):
                zf.write(item_path, item)
            else:
                for root, dirs, fnames in os.walk(item_path):
                    for f in fnames:
                        fp = os.path.join(root, f)
                        zf.write(fp, os.path.relpath(fp, model_dir))

    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"Zipped: {zip_name} ({size_mb:.1f} MB)")
    colab_files.download(zip_name)
    print("Download started!")

Latest checkpoint: checkpoint-576
Zipped: finetuned_parser.zip (1573.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!


In [9]:
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch
import json

model_path = '/content/models/finetuned_parser'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print('Model loaded')

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Model loaded


In [10]:
def predict(input_text):
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, padding='max_length', max_length=256).to(device)
    outputs = model.generate(inputs['input_ids'], max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_cases = [
    'add: gym at 6am',
    'add: meditate every morning',
    'add: finish report by Friday',
    'add: team meeting at 3pm',
]

for inp in test_cases:
    output = predict(inp)
    print(f'Input:  {inp}')
    print(f'Output: {output}')
    print()

Input:  add: gym at 6am
Output: name=gym[EXP] | difficulty=0.47[PRD] | duration=45[PRD] | category=fitness[PRD] | importance=0.44[PRD] | fixed_time=true[EXP] | fixed_start=06:00[EXP] | recurrent=false[EXP]

Input:  add: meditate every morning
Output: name=meditation[EXP] | difficulty=0.37[PRD] | duration=45[PRD] | category=work[PRD] | importance=0.55[PRD] | fixed_time=false[EXP] | recurrent=false[EXP]

Input:  add: finish report by Friday
Output: name=finish report[EXP] | deadline=Friday[EXP] | difficulty=0.47[PRD] | duration=45[PRD] | category=work[PRD] | importance=0.44[PRD] | fixed_time=false[EXP] | recurrent=false[EXP]

Input:  add: team meeting at 3pm
Output: name=team meeting[EXP] | difficulty=0.47[PRD] | duration=45[PRD] | category=work[PRD] | importance=0.44[PRD] | fixed_time=true[EXP] | fixed_start=15:00[EXP] | recurrent=false[EXP]



In [ ]:
print('VM.AI Parser - Interactive Test')
print('Type your prompts (or "quit" to exit)')
print()

while True:
    user_input = input('Enter: ')
    if user_input.lower() == 'quit':
        break

    if not user_input.startswith('add:') and not user_input.startswith('modify:'):
        user_input = 'add: ' + user_input

    output = predict(user_input)
    print(f'Output: {output}')
    print()

VM.AI Parser - Interactive Test
Type your prompts (or "quit" to exit)

Enter: add: go home
Output: name=go home[EXP] | difficulty=0.37[PRD] | duration=45[PRD] | category=home[PRD] | importance=0.55[PRD] | fixed_time=false[EXP] | recurrent=false[EXP]

Enter: modify: at 6 am
Output: fixed_time=true[EXP] | fixed_start=18:00[EXP]

Enter: modify: go to gym at 6 pm, hard session
Output: name=go to gym[EXP] | difficulty=0.4[PRD] | duration=60[PRD] | category=fitness[PRD] | importance=0.4[PRD] | fixed_time=true[EXP] | fixed_start=18:00[EXP] | recurrent=false[EXP]

Enter: add: go to gym at 6 pm for 2 hours very hard
Output: name=go to gym[EXP] | difficulty=0.47[PRD] | duration=60[PRD] | category=fitness[PRD] | importance=0.44[PRD] | fixed_time=true[EXP] | fixed_start=18:00[EXP] | recurrent=false[EXP]

